# scoreboard

> Cross-evaluator scoreboard on an explicit, versioned contract (#108): every field is
> sourced from a named artifact or explicitly unknown — never a fabricated default.

In [ ]:
#| default_exp scoreboard

In [ ]:
#| export
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import structlog

log = structlog.get_logger()

# Bump when a column is added/removed/renamed or its semantics change. Written into
# every row (survives parquet AND csv) so downstream consumers can gate on it.
SCOREBOARD_SCHEMA_VERSION = "2.0"

# Every model the pipeline can run. The scoreboard emits an AUC column for each,
# ALWAYS — NaN when that model did not run. A model not running is normal
# configuration variance, so it is NOT recorded in missing_fields.
MODEL_IDS = ("lr", "rf", "xgb", "tabpfn", "tabpfn_ft", "tabicl", "tabicl_ft")

# The full, fixed column set — identical on every run regardless of which models ran
# or which artifacts were present. Consumers can rely on this schema; a change here
# requires a SCOREBOARD_SCHEMA_VERSION bump.
SCOREBOARD_COLUMNS: tuple[str, ...] = (
    "evaluator",
    "status",
    "error_detail",
    "best_auc",
    "best_model",
    *[f"auc_{m}" for m in MODEL_IDS],
    "cv_folds",
    "nested_cv",
    "n_features",
    "sensitivity",
    "specificity",
    "n_samples",
    "n_positive",
    "sens_at_100spec",
    "sens_at_100spec_healthy",
    "n_detected_at_100spec",
    "sens_at_95spec",
    "best_sens_100spec_healthy",
    "holdout_auc",
    "holdout_sens_100spec",
    "holdout_n_train",
    "holdout_n_test",
    "auc_drop",
    "selection_method",
    "n_selected_features",
    "selection_total_input",
    "missing_fields",
    "n_missing",
    "schema_version",
)

In [ ]:
#| export
def load_selection_qc(directory: Path) -> dict[str, dict]:
    """Load per-evaluator selection-QC sidecars (``<evaluator>_selection_qc.json``).

    The select stage writes these next to the selected matrices
    (``matrices/selected/`` in the published outdir); the Nextflow scoreboard
    process stages them flat into its work dir. Both layouts are searched.

    Returns:
        Dict keyed by evaluator name. Unreadable files are logged and skipped
        (the affected evaluator then carries ``selection_method="unknown"`` plus a
        ``missing_fields`` entry — visible, never defaulted).
    """
    directory = Path(directory)
    out: dict[str, dict] = {}
    seen: set[Path] = set()
    for base in (directory, directory / "matrices" / "selected"):
        if not base.is_dir():
            continue
        for f in sorted(base.glob("*_selection_qc.json")):
            if f.resolve() in seen:
                continue
            seen.add(f.resolve())
            name = f.name.removesuffix("_selection_qc.json")
            try:
                out[name] = json.loads(f.read_text())
            except (json.JSONDecodeError, OSError) as exc:
                log.error("selection_qc_unreadable", file=str(f), error=str(exc))
    return out


In [ ]:
#| export
def extract_evaluator_summary(
    evaluator: str,
    data: dict,
    selection_qc: dict | None = None,
) -> dict:
    """Extract one scoreboard row from a merged model-results dict + QC sidecar.

    THE single place that knows the model-results key shapes for summary fields
    (#108) — report_data consumes the resulting row rather than re-deriving these.

    Contract: every ``SCOREBOARD_COLUMNS`` field is either sourced from a named
    key/artifact or explicitly unknown (NaN / None / ``"unknown"``) **and** listed
    in ``missing_fields``. No plausible-looking defaults: absence must be
    distinguishable from a measured value (fail loud, not to 0.0 — invariant #3).
    """
    rec: dict[str, Any] = {c: None for c in SCOREBOARD_COLUMNS}
    missing: list[str] = []
    rec["evaluator"] = evaluator
    rec["schema_version"] = SCOREBOARD_SCHEMA_VERSION

    # ── Per-model AUCs (fixed column set; absent model = NaN, not missing) ──
    all_aucs: dict[str, float] = {}
    for m in MODEL_IDS:
        v = data.get(f"auc_{m}")
        if v is not None and not (isinstance(v, float) and np.isnan(v)):
            all_aucs[m] = float(v)
            rec[f"auc_{m}"] = float(v)
        else:
            rec[f"auc_{m}"] = np.nan

    rec["best_model"] = max(all_aucs, key=all_aucs.get) if all_aucs else None
    rec["best_auc"] = all_aucs.get(rec["best_model"], np.nan) if all_aucs else np.nan

    # ── Failure status (#59/#60 semantics, unchanged) ──
    error_detail = data.get("error")
    rec["error_detail"] = error_detail
    if error_detail:
        rec["status"] = "PARTIAL" if all_aucs else "FAILED"
    elif not all_aucs:
        rec["status"] = "NO_RESULTS"
    else:
        rec["status"] = "OK"

    def take(column: str, *keys: str, transform=None):
        """Source a field from the first present key, else mark it missing."""
        for k in keys:
            if k in data and data[k] is not None:
                rec[column] = transform(data[k]) if transform else data[k]
                return
        missing.append(column)

    take("cv_folds", "cv_folds_actual")
    take("nested_cv", "nested_cv")  # nullable: True/False/None — never defaulted False
    take("n_features", "top_features", transform=len)

    best = rec["best_model"]
    if best:
        # 0.5-threshold sensitivity/specificity from the classification report
        # (key shape varies: "1" vs "1.0" class labels).
        cr = data.get(f"{best}_classification_report")
        if isinstance(cr, dict) and cr:
            pos = cr.get("1", cr.get("1.0", {})) or {}
            neg = cr.get("0", cr.get("0.0", {})) or {}
            weighted = cr.get("weighted avg", {}) or {}
            rec["sensitivity"] = pos.get("recall", np.nan)
            rec["specificity"] = neg.get("recall", np.nan)
            rec["n_samples"] = weighted.get("support", np.nan)
            rec["n_positive"] = pos.get("support", np.nan)
        else:
            missing.extend(["sensitivity", "specificity", "n_samples", "n_positive"])

        take("sens_at_100spec", f"{best}_sensitivity_at_100spec")
        take("sens_at_100spec_healthy", f"{best}_sensitivity_at_100spec_healthy")
        take("n_detected_at_100spec", f"{best}_n_detected_at_100spec")
        take("sens_at_95spec", f"{best}_sensitivity_at_95spec")
        take("holdout_auc", f"holdout_{best}_auc")
        take("holdout_sens_100spec", f"holdout_{best}_sensitivity_at_100spec")
    else:
        missing.extend(
            [
                "sensitivity",
                "specificity",
                "n_samples",
                "n_positive",
                "sens_at_100spec",
                "sens_at_100spec_healthy",
                "n_detected_at_100spec",
                "sens_at_95spec",
                "holdout_auc",
                "holdout_sens_100spec",
            ]
        )

    take("holdout_n_train", "holdout_n_train")
    take("holdout_n_test", "holdout_n_test")

    ha, ba = rec.get("holdout_auc"), rec.get("best_auc")
    if ha is not None and ba is not None and not pd.isna(ha) and not pd.isna(ba):
        rec["auc_drop"] = ba - ha
    else:
        rec["auc_drop"] = np.nan  # holdout_auc already in missing when unsourced

    # Best sens@100spec_healthy across every model that ran.
    sh_vals = [
        v
        for m in all_aucs
        if (v := data.get(f"{m}_sensitivity_at_100spec_healthy")) is not None
    ]
    if sh_vals:
        rec["best_sens_100spec_healthy"] = max(sh_vals)
    else:
        rec["best_sens_100spec_healthy"] = np.nan
        missing.append("best_sens_100spec_healthy")

    # ── Selection metadata: the QC sidecar is the ONLY source (#107/#108).
    # The legacy embedded-dict convention (and its "legacy_cohens_d" default label)
    # described the deleted monolithic path and produced false metadata on every
    # decomposed run — absence is now visible, never relabeled.
    if selection_qc:
        rec["selection_method"] = selection_qc.get("method", "unknown")
        rec["n_selected_features"] = selection_qc.get(
            "n_mrmr_selected", selection_qc.get("n_selected_union")
        )
        rec["selection_total_input"] = selection_qc.get("total_input_features")
        for col in ("n_selected_features", "selection_total_input"):
            if rec[col] is None:
                missing.append(col)
    else:
        rec["selection_method"] = "unknown"
        missing.extend(
            ["selection_method", "n_selected_features", "selection_total_input"]
        )

    rec["missing_fields"] = ";".join(missing)
    rec["n_missing"] = len(missing)
    return rec


In [ ]:
#| export
def build_scoreboard(output_dir: Path) -> pd.DataFrame:
    """Aggregate all evaluator model results into a ranked scoreboard.

    Scans ``output_dir`` for ``*_model_results.json`` (CPU+GPU, merged per
    evaluator) and ``*_selection_qc.json`` sidecars, and returns one row per
    evaluator with the fixed ``SCOREBOARD_COLUMNS`` schema, sorted by
    ``best_auc`` descending.

    Args:
        output_dir: Directory containing the result JSONs (flat, as staged by
            the Nextflow scoreboard process, or a published outdir — QC sidecars
            are also found under ``matrices/selected/``).

    Returns:
        DataFrame with the full contract schema. Empty DataFrame (with the
        contract columns) if no results are found.
    """
    from kreview.eval_engine import load_all_model_results

    output_dir = Path(output_dir)
    all_results = load_all_model_results(output_dir)
    sel_qc = load_selection_qc(output_dir)

    if not all_results:
        log.warning("scoreboard_no_results", dir=str(output_dir))
        return pd.DataFrame(columns=list(SCOREBOARD_COLUMNS))

    records = []
    for evaluator_name, data in all_results.items():
        try:
            records.append(
                extract_evaluator_summary(
                    evaluator_name, data, sel_qc.get(evaluator_name)
                )
            )
        except Exception as exc:
            # Do NOT drop the evaluator silently (#60 spirit): emit a FAILED row so
            # it is visible in the scoreboard rather than vanishing from the table.
            log.error(
                "scoreboard_evaluator_parse_failed",
                evaluator=evaluator_name,
                error=str(exc),
            )
            rec: dict[str, Any] = {c: None for c in SCOREBOARD_COLUMNS}
            rec.update(
                {
                    "evaluator": evaluator_name,
                    "status": "FAILED",
                    "error_detail": f"scoreboard_parse_error: {exc}",
                    "best_auc": np.nan,
                    "selection_method": "unknown",
                    "missing_fields": "parse_error",
                    "n_missing": len(SCOREBOARD_COLUMNS),
                    "schema_version": SCOREBOARD_SCHEMA_VERSION,
                }
            )
            records.append(rec)
            continue

    df = pd.DataFrame(records, columns=list(SCOREBOARD_COLUMNS)).sort_values(
        "best_auc", ascending=False
    )

    # Surface degraded/failed evaluators loudly (#59/#60), and incomplete-but-OK
    # rows too (#108): a complete-LOOKING row with absent inputs must be visible.
    if len(df):
        n_failed = int((df["status"] == "FAILED").sum())
        n_partial = int((df["status"] == "PARTIAL").sum())
        n_no_results = int((df["status"] == "NO_RESULTS").sum())
        if n_failed or n_partial or n_no_results:
            degraded = df[df["status"] != "OK"]
            log.warning(
                "scoreboard_degraded_evaluators",
                n_failed=n_failed,
                n_partial=n_partial,
                n_no_results=n_no_results,
                evaluators={row.evaluator: row.status for row in degraded.itertuples()},
            )
        incomplete = df[df["n_missing"] > 0]
        if len(incomplete):
            log.warning(
                "scoreboard_missing_summary",
                n_rows_with_missing=int(len(incomplete)),
                worst={
                    row.evaluator: row.missing_fields
                    for row in incomplete.nlargest(5, "n_missing").itertuples()
                },
            )
    log.info(
        "scoreboard_built",
        schema_version=SCOREBOARD_SCHEMA_VERSION,
        n_evaluators=len(df),
        best_evaluator=df.iloc[0]["evaluator"] if len(df) else None,
        best_auc=float(df.iloc[0]["best_auc"]) if len(df) else None,
    )
    return df